# Exercise 2.2.2.1 — implement `VPGAgent`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.2.2 Policy Gradient`  
**Notebook:** `2.2.2_Policy_Gradient_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.2.2.1](https://delta-drills.vercel.app/?arena_exercise=2.2.2.1)


# [2.2.2] - Vanilla Policy Gradient (VPG) (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/21_[2.2.2]_Policy_Gradient)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_Policy_Gradient_exercises.ipynb?t=20250917) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_Policy_Gradient_solutions.ipynb?t=20250917)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

# Introduction

You'll also implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm upon which many modern RL algorithms are based (including PPO).

## Content & Learning Objectives

### 1️⃣ Policy Gradient Theorem

The Policy Gradient Theorem is what all policy gradient methods are based on: it allows us to compute the gradient of the return, something that would naively not have a well defined gradient.

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem

### 2️⃣ Implementation


> ##### Learning Objectives
>
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

## 🚧 Under construction 🚧

This material is still in beta, and may be severely lacking in tests, or have bugs. Please report problems you find in `#errata`!

## Optional Readings

* [Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/) (25 minutes)
    * Skip the derivation of the policy gradient theorem, we've already done that here.
    * Covers many other policy gradient methods we don't, you may wish to implement some of them afterwards as a bonus.

## Setup code

In [ ]:
from __future__ import annotations

import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, TypeAlias, Optional

import gymnasium as gym
import numpy as np
import torch as t
import wandb
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from tqdm import tqdm, trange
import torch.nn.functional as F
from torch.utils.data import DataLoader

from eindex import eindex

warnings.filterwarnings("ignore")

ActType = Int
ObsType = Int

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part21_dqn"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part22_vpg.tests as tests
import part22_vpg.utils as utils
from part1_intro_to_rl.solutions import Environment, Norvig, Toy, find_optimal_policy
from part1_intro_to_rl.utils import set_global_seeds
from rl_utils import make_env
from plotly_utils import cliffwalk_imshow, line, plot_cartpole_obs_and_dones
from rl_utils import generate_and_plot_trajectory


from gpu_env import CartPole
from probe import Probe4, Probe5
from collections import namedtuple
from torch.utils.data import Dataset, TensorDataset

from torchinfo import summary


device = t.device(
   "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)

MAIN = __name__ == "__main__"

# 1️⃣ Policy Gradient Theorem

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem

Instead of learning action-values and deriving a policy (as in Q-learning or DQN), **policy gradient methods learn the policy directly**.  
- Policy is parameterized: $\pi_\theta(a|s)$ with parameters $\theta$ (often a neural network).  
- Objective: Choose $\theta$ to maximize expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$ (joy), where $\tau$ is a trajectory and $G(\tau)$ its return.  

We would desire to update the policy directly via **gradient ascent** against $J(\theta)$:
$$
\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)
$$

The problem is that the return is a sum of rewards from the trajectory, and the trajectory itself is a result of sampling from the policy, over and over, 
as well as being dependant on the environmental distribution, which we do not have access to.
There is no clear way to directly compute the gradient of the return with respect to the policy parameters.
The solution here is the **policy gradient theorem**, which states that we can instead use the log-probability weighted return as an unbiased estimator of the gradient of the return.

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t) \right]
$$

<details>
<summary>Derivation</summary>

The probability of sampling a trajectory 
$\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ 
is given by
$$
\Pr(\tau|\theta) = \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t)
$$
where $\mu$ is the environment transition probability.

$$
\begin{align*}
   \nabla_\theta J(\theta) &= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)] \\
   &= \nabla_\theta \sum_\tau  \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \nabla_\theta \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \Pr(\tau|\theta)\,\nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \\
   &= \mathbb{E}_{\tau \sim \pi_\theta}\left[ \nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \right]
   \end{align*}
   $$
   where we made use of the log-derivative trick: $\nabla_\theta p(x) = p(x) \nabla_\theta \log p(x)$.
  
 
   The dynamics $\mu$ do not depend on $\theta$, so:
   $$
   \begin{align*}
   \log \Pr(\tau|\theta) &= \log \left( \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t) \right) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \sum_{t=0}^{T-1} \log \mu(s_{t+1}|s_t, a_t) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \text{const.}
   \end{align*}
   $$
   where the const. term is independent of $\theta$, so when we take the gradient, it vanishes.

   Thus:
   $$
   \nabla_\theta \log \Pr(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
   $$

Plugging back into the gradient:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G(\tau)
\right]
$$

This is the **Vanilla Policy Gradient estimator**, also called **REINFORCE**. 
Each $\log \pi_\theta(a_t|s_t)$ is multiplied by the **full return** $G(\tau)$. 
However, the action $a_t$ cannot influence rewards before time $t$, only those afterwards.
This means that all the rewards before timestep $t$ merely add noise, as no changes to the policy
can affect them.  To reduce variance, replace $G(\tau)$ with the return $G_t$ at timestep $t$, 
also called the **reward-to-go**:
$$
G_t = \sum_{i=t}^{T} \gamma^{i-t} r_{i}
$$

Thus, the lower-variance unbiased estimator is:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G_t
\right]
$$

</details>

There are many other variants of the policy gradient estimator, as described in [Schulman, 2018](https://arxiv.org/abs/1506.02438).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png" width="800">

# 2️⃣ Implementation

> ##### Learning Objectives
>
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

We make use of the same CartPole environment as before, but now we have a vectorized version that is entirely defined in terms of tensor operations (see `chapter2_rl/exercises/gpu_env.py`). This environment is identical to the one used for DQN, but it now runs entirely on the GPU. This means
* we don't need to constantly convert between numpy and torch tensors
* we can run large numbers of environments in parallel (~thousands of environments for ~millions of environmental steps per second)
* we avoid copying data back and forth between the CPU and GPU, which can be a significant bottleneck

## Policy Network

Here, the policy is learned directly as a neural network, rather than learning a Q-value table approximator. We'll use the same architecture as the Q-network from DQN, so we've just included that here for you.

In [ ]:
class PolicyNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential


    def __init__(
        self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]
    ):
        super().__init__()
        #assert len(obs_shape) == 1, f"Expecting a single vector of observations, got {obs_shape}"
        assert len(hidden_sizes) == 2, f"Expecting 2 hidden layers, got {len(hidden_sizes)}"
        self.layers = nn.Sequential(nn.Linear(obs_shape[-1], hidden_sizes[0]),
                                    nn.ReLU(),
                                    nn.Linear(hidden_sizes[0], hidden_sizes[1]),
                                    nn.ReLU(),
                                    nn.Linear(hidden_sizes[1], num_actions))

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)

net = PolicyNetwork(obs_shape=(4,), num_actions=2)
summary(net)

## Rollout Buffer

The way that our implementation of VPG will work is simple: we perform a rollout acrosss `num_envs` many environments in parallel, and store the trajectories for each. We then learn from that set of rollouts, and then discard it afterwards. One rollout, one learning step. This means we are always learning **on-policy**: we only every learn from data that the current model actually generated. We will use a rollout buffer to store the trajectories.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.2.2.1"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part22_vpg.solutions import Rollout


### Exercise - implement `VPGAgent`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Implement the functions:
* `gen_rollout` - this function compute the episode rollout, by interacting with the environment for `args.num_steps_per_rollout` steps. If an episode terminates, we reset the environment and continue. We will track the length of the episode in the `lifespan` variable, which indicates how long each episode runs before termination. FOr the cartpole environment, this will allow us to track performance (the longer the cart lives, the better it does.)

* `get_actions` - this function takes in an observation, and returns the actions, logprobs, and entropy for that observation. You can use `t.distributions.Categorical(logits=logits)` to construct a distribution, from which you can get the actions, logprobs, and entropy. [See the docs](https://docs.pytorch.org/docs/stable/distributions.html#torch.distributions.categorical.Categorical) for details.

In [ ]:
class VPGAgent:
    """Base Agent class handling the interaction with the environment."""

    dead : Bool[Tensor, " num_envs"]
    lifespan : Int[Tensor, " num_envs"]


    def __init__(
        self,
        envs: gym.Env,
        policy_network: PolicyNetwork,
        args: VPGArgs,
        rng: Optional[np.random.Generator] = None,
    ):
        self.envs = envs
        self.policy_network = policy_network
        self.rng = rng
        self.args = args
        self.obs_shape = envs.observation_space.shape
        self.action_shape = envs.action_space.shape

    @t.no_grad()
    def gen_rollout(self, rollout: Rollout) -> tuple[Rollout, dict[str, Any]]:
        """
        Compute the full episode rollout for all environments in parallel, adding them to the rollout buffer.
        It then returns the rollout buffer, and a dictionary of info contining the lifespan.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        obs, _ = self.envs.reset()  # Need a starting observation
        device = self.args.device   
        
        dead = t.zeros(self.args.num_envs, dtype=t.bool, device=device)
        lifespan = t.zeros(self.args.num_envs, dtype=t.int32, device=device)
        rollout.reset()
        
        raise NotImplementedError()
        
        info = {"lifespan": lifespan}
        
        return rollout, info

    def get_actions(self, obs: Float[Tensor, " num_envs *obs_shape"]
    ) -> tuple[Int[Tensor, " num_envs *action_shape"], 
               Float[Tensor, " num_envs"],
               Float[Tensor, " num_envs"]]:
        """
        Computes the agents turn: given an observation for eahc environment,
        sample the action the agent takes, along with the log_probs of that action,
        and the entropy of the action distribution.
        """
        raise NotImplementedError()

<details><summary>Solution</summary>

```python
class VPGAgent:
    """Base Agent class handling the interaction with the environment."""

    dead : Bool[Tensor, " num_envs"]
    lifespan : Int[Tensor, " num_envs"]


    def __init__(
        self,
        envs: gym.Env,
        policy_network: PolicyNetwork,
        args: VPGArgs,
        rng: Optional[np.random.Generator] = None,
    ):
        self.envs = envs
        self.policy_network = policy_network
        self.rng = rng
        self.args = args
        self.obs_shape = envs.observation_space.shape
        self.action_shape = envs.action_space.shape

    @t.no_grad()
    def gen_rollout(self, rollout: Rollout) -> tuple[Rollout, dict[str, Any]]:
        """
        Compute the full episode rollout for all environments in parallel, adding them to the rollout buffer.
        It then returns the rollout buffer, and a dictionary of info contining the lifespan.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        obs, _ = self.envs.reset()  # Need a starting observation
        device = self.args.device   
        
        dead = t.zeros(self.args.num_envs, dtype=t.bool, device=device)
        lifespan = t.zeros(self.args.num_envs, dtype=t.int32, device=device)
        rollout.reset()
        
        for timestep in range(self.args.num_steps_per_rollout):
        
            actions, logprobs, entropy = self.get_actions(obs)
            new_obs, rewards, terminates, _, info = self.envs.step(actions)
            done = terminates
            rollout.add_step(obs, actions, logprobs, rewards, done, info)
            obs = new_obs
            dead = dead | done
            lifespan += ~dead
        
        info = {"lifespan": lifespan}
        
        return rollout, info

    def get_actions(self, obs: Float[Tensor, " num_envs *obs_shape"]
    ) -> tuple[Int[Tensor, " num_envs *action_shape"], 
               Float[Tensor, " num_envs"],
               Float[Tensor, " num_envs"]]:
        """
        Computes the agents turn: given an observation for eahc environment,
        sample the action the agent takes, along with the log_probs of that action,
        and the entropy of the action distribution.
        """
        logits = self.policy_network(obs)
        dist = t.distributions.Categorical(logits=logits)
        actions = dist.sample()
        entropy = dist.entropy()
        logprobs = dist.log_prob(actions)
        return actions, logprobs, entropy
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
